# Upper Confidence Bound (UCB) based agents

> Agents utelizing the UCB based approach for Dynamic pricing and learning problems from https://doi.org/10.48550/arXiv.1604.07463

In [ ]:
#| default_exp agents.dynamic_pricing.UCB

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging

from abc import ABC, abstractmethod
from typing import Union, Optional, List
import numpy as np
import os
from ddopai.agents.dynamic_pricing.utils import GLMLink
from ddopai.envs.base import BaseEnvironment
from ddopai.agents.dynamic_pricing.mushroom_rl import PricingMushroomBaseAgent
from mushroom_rl.core import Agent
from ddopai.utils import MDPInfo
from ddopai.agents.obsprocessors import FlattenTimeDimNumpy
from ddopai.envs.actionprocessors import ClipAction
from scipy.optimize import minimize

In [ ]:
#| export
class UCBPolicy:
    def __init__(self,
                 lam: float,
                 reg: float,
                 environment_info: MDPInfo,
                 obsprocessors=None,
                 actionprocessors=None,
                 agent_name=None,
                 ex_prices=None,
                 alpha=None,
                 beta=None,
                 price_function=None,
                 g=None):

        if alpha is None:
            alpha = np.zeros(environment_info.observation_space['features'].shape[0])
        if beta is None:
            beta = np.zeros(environment_info.observation_space['features'].shape[0])
        if isinstance(ex_prices, list):
            ex_prices = np.array(ex_prices)
        assert ex_prices.shape[0] >= 2

        self.environment_info = environment_info
        self.ex_prices = ex_prices
        self.alpha = alpha
        self.beta = beta
        self.actionprocessors = actionprocessors or []
        self.obsprocessors = obsprocessors or []
        self.price_function = price_function
        self.g = g
        self.lam = lam
        self.reg = reg
        self.t = 0
        self.X = np.empty((0, environment_info.observation_space['features'].shape[0] * 2))
        self.Y = np.empty((0, 1))
        self.mode = "train"
        self.actionprocessors.append(ClipAction(environment_info.action_space.low, environment_info.action_space.high))
        self.d = environment_info.observation_space['features'].shape[0]
    def draw_action(self, observation):
        x = observation['features']
        # if self.t in [0, 1]:
        #     price = self.ex_prices[self.t]
        # else:
        M = self.sample_design_matrix()
        samples = self.sample_from_confidence_region(np.concatenate([self.alpha, self.beta]), M)
        alpha, beta = self.max_rev(samples, x)
        price = self.price_function(x, alpha, beta)
        for processor in self.actionprocessors:
            price = processor(price)
        return np.array(price, dtype=np.float32)

    def fit(self, X, Y, action):

        Z = np.concatenate([X, X * action])
        self.X = np.vstack([self.X, Z])
        self.Y = np.vstack([self.Y, Y])
        self.parameter_update(Z, Y)
        self.t += 1

    def parameter_update(self, z, D_t):
        """
        One-step Sherman-Morrison update of the quasi-MLE for the *identity* link g(u)=u
        (linear demand).  If you keep a general g, replace D_t by the *score* below.
        """
        if self.t == 0:
            # first call: initialise
            d = len(z)
            self.M_inv = np.eye(d) / self.lam   if self.lam != 0 else np.eye(d)
            self.q = np.zeros(d)

        # rank-1 update of M_t^{-1}
        Mz = self.M_inv @ z
        self.M_inv -= np.outer(Mz, Mz) / (1.0 + z @ Mz)

        # running first-order term
        self.q += z * float(D_t)
        # new parameter
        theta_hat = self.M_inv @ self.q
        d = theta_hat.size // 2
        self.alpha, self.beta = theta_hat[:d], theta_hat[d:]


    def sample_design_matrix(self):
        # d = self.environment_info.observation_space['features'].shape[0]
        # I = self.lam * np.identity(2 * d)
        # if self.X.shape[0] == 0:
        #     return I
        # return I + self.X.T @ self.X
        return np.linalg.inv(self.M_inv)

    def sample_from_confidence_region(self, theta_hat, M, N=50, gamma=None):
        """
        Draw N points uniformly at random from
            {theta : (theta - theta_hat)^T M^{-1} (theta - theta_hat) <= gamma}
        """
        d = len(theta_hat)                        # here d = 2·feature_dim
        if gamma is None:
            # Simple hard-coded radius like the authors’ demo: Γ = d / 20
            # In production you would compute the analytic β_t²
            gamma = d / 20.0

        # Cholesky factor of M^{-1}
        L = np.linalg.cholesky(np.linalg.inv(M))

        # 1. Draw points *in* the unit ball (not only on the surface)
        rng = np.random.default_rng()
        u = rng.normal(size=(d, N))
        u /= np.linalg.norm(u, axis=0)            # on the sphere
        r = rng.random(N)**(1.0 / d)              # radii ∼ U[0,1]^{1/d}
        u *= r                                    # now in the ball

        # 2. Map ball → ellipsoid and 3. translate by theta_hat
        samples = theta_hat[:, None] + np.sqrt(gamma) * (L @ u)
        return samples.T                          # shape (N, d)


    def max_rev(self, samples, x):
        max_val = -np.inf
        best_alpha, best_beta = None, None
        for theta in samples:
            alpha = theta[:x.shape[0]]
            beta = theta[x.shape[0]:]
            a = np.dot(x, alpha)
            b = np.dot(x, beta)
            b = min(-0.01, b)
            a = max( 0.01, a)
            price = self.price_function(np.ones_like(x), a, b)

            rev = price * self.g.g(a + price * b)
            if rev > max_val:
                max_val = rev
                best_alpha, best_beta = alpha, beta
        return best_alpha, best_beta

    def update_task(self, env):
        self.environment_info = env.mdp_info
        self.d = self.environment_info.observation_space['features'].shape[0]
        self.X = np.empty((0, 2 * self.d))
        self.Y = np.empty((0, 1))
        self.actionprocessors[-1] = ClipAction(self.environment_info.action_space.low, self.environment_info.action_space.high)
        self.t = 0

    def reset(self):
        pass


In [ ]:
#| export
class UCBCoreAgent(Agent):

    """
    Base class for UCB agents.
    """

    def __init__(self,
                 lam: float,
                 reg: float,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] = [],
                 actionprocessors: Optional[List[object]] = [],
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 alpha: np.ndarray | None = None,
                 beta: np.ndarray | None = None,
                 price_function = None,
                 g = None,
                 ):
        
        policy = UCBPolicy(lam=lam, reg=reg, environment_info=environment_info, obsprocessors=obsprocessors, actionprocessors=actionprocessors, ex_prices=ex_prices, alpha=alpha, beta=beta, price_function=price_function, g=g)
        self.agent_name = agent_name
        super().__init__(environment_info, policy)
        
    def fit(self, dataset, **kwargs):
        X = dataset[0][0]['features']
        Y = kwargs["demand"][0]
        action = dataset[0][1]
        self.policy.fit(X, Y, action)
        
    def update_task(self, env):
        self.policy.update_task(env)

In [ ]:
#| export
class UCBAgent(PricingMushroomBaseAgent):
    """
    Wrapper class for UCBCoreAgent to interact with MushroomRL.
    """
    def __init__(self,
                 lam: float,
                 reg: float,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] =[],
                 actionprocessors: Optional[List[object]] = [],
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 alpha: np.ndarray | None = None,
                 beta: np.ndarray | None = None,
                 price_function = None,
                 g = None,
                 ):
        self.agent = UCBCoreAgent(lam=lam, reg=reg, environment_info=environment_info,
                                  obsprocessors=obsprocessors, 
                                  actionprocessors=actionprocessors, 
                                  agent_name=agent_name, 
                                  ex_prices=ex_prices, 
                                  alpha=alpha, 
                                  beta=beta, 
                                  price_function=price_function, 
                                  g=g)
        super().__init__(environment_info=environment_info, obsprocessors=obsprocessors, agent_name=agent_name)
    def update_task(self, env: object):
        """ Update the environment specific parameters of the agent """
        self.agent.update_task(env)